# Stage 9 - Evaluate (gold set only)

Every number here comes from the human gold set. Confidence intervals and comparisons use a **conversation bootstrap** (1,000 resamples). Runs not trained yet are listed as missing. Settings: `configs/evaluate.yaml`. Outputs: `results/eval/`. CPU is enough.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.evaluate.metrics import evaluate_runs

summary = evaluate_runs("evaluate")
print("evaluated:", summary["runs_evaluated"], "| missing:", summary["runs_missing"])

## Results table (for the report)

In [ ]:
import pandas as pd
from krs.config import RESULTS_DIR

ev = RESULTS_DIR / "eval"
pd.read_csv(ev / "results_table.csv")

## Hypotheses

- **H1** M1 beats B1 and B2: `a_better` = the 95% CI of the difference is above 0
- **H2** M1 within 5 F1 points of the teacher: `within_margin` = the CI lower bound of M1 − T is above −0.05
- **H3** context helps: M1 vs A1

In [ ]:
rows = []
for c in summary["comparisons"]:
    for axis in ("knowledge_types", "domains"):
        r = c.get(axis)
        if r:
            rows.append({"hypothesis": c["hypothesis"], "a": c["a"], "b": c["b"], "axis": axis, **r})
        elif axis == "knowledge_types":
            rows.append({"hypothesis": c["hypothesis"], "a": c["a"], "b": c["b"], "axis": "-", "status": c.get("status")})
pd.DataFrame(rows)

## Per-label scores and per-source breakdown (M1)

In [ ]:
import json

m1 = json.loads((ev / "report_M1.json").read_text())
pd.DataFrame(m1["knowledge_types"]["per_label"])

In [ ]:
pd.DataFrame(m1["domains"]["per_label"])

In [ ]:
pd.DataFrame({a: m1[a]["macro_f1_by_source"] for a in ("knowledge_types", "domains")})

## Confusion matrices and learning curve

In [ ]:
from IPython.display import Image, display

for f in sorted(ev.glob("confusion_*.png")) + sorted(ev.glob("learning_curve.png")):
    display(Image(filename=str(f)))